In [1]:
#from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
from imutils.video import VideoStream
import numpy as np
import argparse
import imutils
import time
import cv2
import os


In [5]:
import numpy as np
print(np.__version__)

1.24.4


In [5]:
!pip install imutils 

In [6]:
!pip install opencv-python 

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
    --------------------------------------- 0.5/40.2 MB 2.8 MB/s eta 0:00:15
   - -------------------------------------- 1.0/40.2 MB 2.5 MB/s eta 0:00:16
   - -------------------------------------- 1.6/40.2 MB 2.6 MB/s eta 0:00:15
   -- ------------------------------------- 2.1/40.2 MB 2.7 MB/s eta 0:00:15
   -- ------------------------------------- 2.9/40.2 MB 2.7 MB/s eta 0:00:14
   --- ------------------------------------ 3.4/40.2 MB 2.8 MB/s eta 0:00:14
   ---- ----------------------------------- 4.2/40.2 MB 2.9 MB/s eta 0:00:13
   ---- ----------------------------------- 4.7/40.2 MB 2.9 MB/s eta 0:00:13
   ----- ---------------------------------- 5.2/40.2 MB 2.8 MB/s eta 0:00:13
   ----- ---------------------------------- 5.8/40.2 MB 2.8 MB/s eta 0:00:13
   ----- -

In [8]:
!pip uninstall numpy -y

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6


You can safely remove it manually.
You can safely remove it manually.


In [9]:
!pip install numpy==1.24.4

   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/14.8 MB 1.4 MB/s eta 0:00:11
   -- ------------------------------------- 0.8/14.8 MB 1.6 MB/s eta 0:00:09
   -- ------------------------------------- 1.0/14.8 MB 1.6 MB/s eta 0:00:09
   ---- ----------------------------------- 1.6/14.8 MB 1.6 MB/s eta 0:00:09
   ---- ----------------------------------- 1.8/14.8 MB 1.7 MB/s eta 0:00:08
   ------ --------------------------------- 2.4/14.8 MB 1.8 MB/s eta 0:00:07
   ------- -------------------------------- 2.6/14.8 MB 1.8 MB/s eta 0:00:07
   -------- ------------------------------- 3.1/14.8 MB 1.8 MB/s eta 0:00:07
   --------- ------------------------------ 3.7/14.8 MB 1.9 MB/s eta 0:00:06
   ----------- ---------------------------- 4.2/14.8 MB 2.0 MB/s eta 0:00:06
   ------------ ----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.24.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.24.4 which is incompatible.
tensorflow-intel 2.13.0 requires numpy<=1.24.3,>=1.22, but you have numpy 1.24.4 which is incompatible.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.15.0 which is incompatible.


In [11]:
!pip uninstall opencv-python opencv-contrib-python -y

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92


In [12]:
!pip install opencv-python==4.8.1.78

   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.1 MB ? eta -:--:--
    --------------------------------------- 0.8/38.1 MB 2.2 MB/s eta 0:00:17
   - -------------------------------------- 1.3/38.1 MB 2.3 MB/s eta 0:00:16
   - -------------------------------------- 1.8/38.1 MB 2.5 MB/s eta 0:00:15
   -- ------------------------------------- 2.4/38.1 MB 2.6 MB/s eta 0:00:14
   --- ------------------------------------ 3.1/38.1 MB 2.7 MB/s eta 0:00:13
   --- ------------------------------------ 3.7/38.1 MB 2.8 MB/s eta 0:00:13
   ---- ----------------------------------- 4.5/38.1 MB 2.9 MB/s eta 0:00:12
   ----- ---------------------------------- 5.0/38.1 MB 2.8 MB/s eta 0:00:12
   ----- ---------------------------------- 5.5/38.1 MB 2.8 MB/s eta 0:00:12
   ------ --------------------------------- 6.3/38.1 MB 2.9 MB/s eta 0:00:11
   ------- -------------------------------- 6.8/38.1 MB 2.9 MB/s eta 0:00:11
   ------- --

In [ ]:
def detect_and_predict_mask(frame, faceNet, emotionNet):
    # grab the dimensions of the frame and then construct a blob
    # from it
    (h, w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300),
        (104.0, 177.0, 123.0))

    # pass the blob through the network and obtain the face detections
    faceNet.setInput(blob)
    detections = faceNet.forward()

    # initialize our list of faces, their corresponding locations,
    # and the list of predictions from our face mask network
    faces = []
    locs = []
    preds = []

    # loop over the detections
    for i in range(0, detections.shape[2]):
        # extract the confidence (i.e., probability) associated with
        # the detection
        confidence = detections[0, 0, i, 2]

        # filter out weak detections by ensuring the confidence is
        # greater than the minimum confidence
        if confidence > args["confidence"]:
            # compute the (x, y)-coordinates of the bounding box for
            # the object
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")

            # ensure the bounding boxes fall within the dimensions of
            # the frame
            (startX, startY) = (max(0, startX), max(0, startY))
            (endX, endY) = (min(w - 1, endX), min(h - 1, endY))

            # extract the face ROI, convert it from BGR to RGB channel
            # ordering, resize it to 224x224, and preprocess it
            face = frame[startY:endY, startX:endX]
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            face = cv2.resize(face, (224, 224))
            face = img_to_array(face)
            face = preprocess_input(face)

            # add the face and bounding boxes to their respective
            # lists
            faces.append(face)
            locs.append((startX, startY, endX, endY))

    # only make a predictions if at least one face was detected
    if len(faces) > 0:
        # for faster inference we'll make batch predictions on *all*
        # faces at the same time rather than one-by-one predictions
        # in the above `for` loop
        faces = np.array(faces, dtype="float32")
        preds =emotionNet.predict(faces, batch_size=32)

    # return a 2-tuple of the face locations and their corresponding
    # locations
    return (locs, preds)

# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()
ap.add_argument("-f", "--face", type=str,
    default="face_detector",
    help="path to face detector model directory")
ap.add_argument("-m", "--model", type=str,
    default="emotion_detector.model",
    help="path to trained face mask detector model")
ap.add_argument("-c", "--confidence", type=float, default=0.5,
    help="minimum probability to filter weak detections")
#args = vars(ap.parse_args())
args = vars(ap.parse_args(args=[]))


# load our serialized face detector model from disk
print("[INFO] loading face emotion detector model...")
#prototxtPath = os.path.sep.join([args["face"],r"C:\hope\Deep Learning\Face_Mask_Project\deploy.prototxt"])

prototxtPath = r"C:\hope\Deep Learning\Face_Emotion_Project\deploy.prototxt"
#weightsPath = os.path.sep.join([args["face"],
   # r"C:\hope\Deep Learning\Face_Mask_Project\res10_300x300_ssd_iter_140000.caffemodel"])
weightsPath=r"C:\hope\Deep Learning\Face_Emotion_Project\res10_300x300_ssd_iter_140000.caffemodel"
faceNet = cv2.dnn.readNet(prototxtPath, weightsPath)

# load the face mask detector model from disk
print("[INFO] loading face emotion detector model...")
emotionNet = load_model(args["model"])

# initialize the video stream and allow the camera sensor to warm up
print("[INFO] starting video stream...")
vs = VideoStream(src=0).start()
time.sleep(2.0)

# loop over the frames from the video stream
while True:
    # grab the frame from the threaded video stream and resize it
    # to have a maximum width of 400 pixels
    frame = vs.read()
    frame = imutils.resize(frame, width=400)

    # detect faces in the frame and determine if they are wearing a
    # face mask or not
    (locs, preds) = detect_and_predict_mask(frame, faceNet,emotionNet)

    # loop over the detected face locations and their corresponding
    # locations
    for (box, pred) in zip(locs, preds):
        # unpack the bounding box and predictions
        (startX, startY, endX, endY) = box
        (happy, sad) = pred

        # determine the class label and color we'll use to draw
        # the bounding box and text
        label = "happy" if happy >sad else "sad"
        color = (0,255,0) if label == "happy" else (0,0,255)

        # include the probability in the label
        #label = "{}: {:.2f}%".format(label, max(mask, withoutMask) * 100)
        
        # display the label and bounding box rectangle on the output
        # frame
        if(label=="happy"):
            
            cv2.putText(frame,"Happy:Welcom!Enjoy your day.", (startX, startY - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
            cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)
        elif(label=="sad"):
            lab="sad:Better Days Ahead."
            cv2.putText(frame, lab, (startX, startY - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
            cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)

    # show the output frame
    cv2.imshow("Frame", frame)
    key = cv2.waitKey(1) & 0xFF

    # if the `q` key was pressed, break from the loop
    if key == ord("q"):
        break

# do a bit of cleanup
cv2.destroyAllWindows()

vs.release()


[INFO] loading face emotion detector model...
[INFO] loading face emotion detector model...
[INFO] starting video stream...
1/1 [==============================] - 0s 82ms/step
